# Multi-label Classification: UCI Yeast Dataset

## Problem Statement
Implement a feedforward neural network for **multi-label classification** using the **UCI Yeast dataset**. This dataset contains information about protein localization sites in yeast cells, where each protein can belong to multiple localization sites simultaneously.

### Dataset Information
- **Source**: UCI Machine Learning Repository - Yeast Dataset
- **Features**: 8 numerical features describing various protein characteristics
- **Labels**: Multiple binary labels representing different cellular localization sites
- **Task**: Predict which localization site(s) each protein belongs to

### Requirements
You must complete the following steps:

1. **Data Acquisition**: Load the UCI Yeast dataset
2. **Data Wrangling**: Preprocess features and convert labels to multi-label format
3. **Model Implementation**: Build a feedforward neural network with sigmoid activation for multi-label output
4. **Training**: Train the model using appropriate loss function and optimizer
5. **Evaluation**: Calculate F1-score and ROC-AUC per label

### Key Considerations
- Multi-label classification requires **sigmoid activation** (not softmax) since labels are non-mutually-exclusive
- Use **Binary Cross-Entropy Loss** for training
- Evaluate each label independently using F1-score and ROC-AUC

<details>
  <summary>💡 Hint</summary>
  Use pandas to load the dataset from UCI repository or local file. Convert class labels to binary format using MultiLabelBinarizer or manual encoding. Design a neural network with sigmoid output layer. Use BCEWithLogitsLoss or BCELoss for training. Calculate metrics per label using sklearn.metrics.
</details>

## Step 1: Data Acquisition

Load the UCI Yeast dataset. The dataset can be obtained from:
- UCI ML Repository: https://archive.ics.uci.edu/ml/datasets/Yeast
- Direct link: https://archive.ics.uci.edu/ml/machine-learning-databases/yeast/yeast.data

The dataset contains:
- Sequence Name
- 8 numerical features (mcg, gvh, alm, mit, erl, pox, vac, nuc)
- Localization site (class label)

In [6]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelBinarizer
from sklearn.metrics import f1_score, roc_auc_score
import warnings
warnings.filterwarnings('ignore')

# TODO: Load the yeast dataset
# Hint: Use pd.read_csv() with appropriate parameters
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/yeast/yeast.data'
column_names = ['seq_name', 'mcg', 'gvh', 'alm', 'mit', 'erl', 'pox', 'vac', 'nuc', 'class']
df = pd.read_csv(url, sep='\\s+', names=column_names, header=None)


In [7]:
print(f"Shape {df.shape}")
print(f"Class distribution:\n {df['class'].value_counts()}")

Shape (1484, 10)
Class distribution:
 class
CYT    463
NUC    429
MIT    244
ME3    163
ME2     51
ME1     44
EXC     35
VAC     30
POX     20
ERL      5
Name: count, dtype: int64


## Step 2: Data Wrangling

Preprocess the dataset:
1. Extract features (numerical columns)
2. Convert single-label format to multi-label binary format
3. Standardize features
4. Split into train and test sets

In [8]:
# TODO: Extract features and labels
X = df.drop(['seq_name', 'class'], axis=1).values
y_labels = df['class'].values

# TODO: Convert labels to multi-label binary format
lb = LabelBinarizer()
y = lb.fit_transform(y_labels)

# TODO: Standardize features
scaler = StandardScaler()
X = scaler.fit_transform(X)

# TODO: Split data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size = 0.2, random_state=42, stratify=y_labels
)

## Step 3: Model Implementation

Build a feedforward neural network with:
- Input layer matching feature dimensions
- Hidden layers with ReLU activation
- Output layer with **sigmoid activation** for multi-label classification

**Important**: Use sigmoid (not softmax) because labels are non-mutually-exclusive.

In [9]:
class MultiLabelNN(nn.Module):
    def __init__(self, input_size, hidden_size, num_labels):
        # TODO: Define layers
        super().__init__()

        self.fc1 = nn.Linear(input_size, hidden_size)
        self.fc2 = nn.Linear(input_size, hidden_size)
        self.fc3 = nn.Linear(input_size, num_labels)

        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()
        self.dropout = nn.Dropout(0.3)
        
    
    def forward(self, x):
        # TODO: Implement forward pass with sigmoid output
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.relu(self.fc2(x))
        x = self.dropout(x)
        return self.fc3(x)
        # x = self.sigmoid() - don't apply the sigmoid, keep gradient information
        # return x

input_size = X_train.shape[1]
hidden_size = 64
num_labels = y_train.shape[1]
model = MultiLabelNN(input_size, hidden_size, num_labels)

print("Model architecture:")
print(model)
print(f"Parameters {sum(p.numel() for p in model.parameters())}")

Model architecture:
MultiLabelNN(
  (fc1): Linear(in_features=8, out_features=64, bias=True)
  (fc2): Linear(in_features=8, out_features=64, bias=True)
  (fc3): Linear(in_features=8, out_features=10, bias=True)
  (relu): ReLU()
  (sigmoid): Sigmoid()
  (dropout): Dropout(p=0.3, inplace=False)
)
Parameters 1242


## Step 4: Training

Train the model using:
- **Loss Function**: Binary Cross-Entropy Loss (BCELoss or BCEWithLogitsLoss)
- **Optimizer**: Adam optimizer
- Monitor training loss over epochs

In [ ]:
# TODO: Initialize model, loss function, and optimizer
# BCELoss for multi-label classification
num_pos = y_train.sum().item()
num_neg = len(y_train) - num_pos
pos_weight =  torch.tensor(num_neg / max(num_pos, 1.0))
criterion = nn.BCELossWithLogits(pos_weight=pos_weight)


# TODO: Implement training loop
# TODO: Track and print training loss

## Step 5: Evaluation

Evaluate the model using:
- **F1-Score** (per label and average)
- **ROC-AUC** (per label and average)

These metrics should be calculated for each label independently since this is multi-label classification.

In [ ]:
# TODO: Make predictions on test set
# TODO: Calculate F1-score per label
# TODO: Calculate ROC-AUC per label
# TODO: Display results